## Benchmark 2: Feynman Subset

In [ ]:
from jaxkan.models.KAN import KAN
from jaxkan.grids import (
    AdaptationState,
    update_state,
    reset_after_adaptation,
    UniformDensity,
    CurvatureDensity,
    MixedAdaptation,
    ScheduledTrigger
)

from benchmarks.feynman import *

from sklearn.model_selection import train_test_split

import jax
import jax.numpy as jnp
from flax import nnx
import optax

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os
import time
from datetime import datetime

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
def generate_feyn_data(function, dim, N, seed):
    key = jax.random.key(seed)

    eps = 1e-6
    x = jax.random.uniform(key, shape=(N,dim), minval=-1.0+eps, maxval=1.0)
    x = jnp.where(jnp.abs(x) < eps, jnp.sign(x) * eps, x)

    y = function(x)

    return x, y

def feyn_fit_eval(model, function, dim):
    if dim == 1:
        resolution = 1000
    elif dim == 2:
        resolution = 200
    elif dim == 3:
        resolution = 30
    else:
        resolution = 10
        
    # Create grid
    eps = 1e-6
    lin = jnp.linspace(-1.0+eps, 1.0, resolution)
    lin = jnp.where(jnp.abs(lin) < eps, jnp.sign(lin) * eps, lin)
    
    xx = jnp.meshgrid(*[lin]*dim)
    grid = jnp.stack([x.ravel() for x in xx], axis=-1)

    # Evaluate ground truth and prediction
    y_true = function(grid)
    y_pred = model(grid)

    # Compute relative L2 error
    error = jnp.linalg.norm(y_true - y_pred) / jnp.linalg.norm(y_true)

    return error

## Training Functions

In [ ]:
@nnx.jit
def train_step(model, optimizer, X_batch, y_batch):
    
    def loss_fn(model):
        residual = model(X_batch) - y_batch
        loss = jnp.mean(residual**2)
        return loss
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    
    return loss

@nnx.jit
def compute_curvature(model, X):
    
    epsilon=1e-3
    n_in = X.shape[1]
    curvatures = jnp.zeros(X.shape[0])
    
    for dim in range(n_in):
        h = jnp.zeros((1, n_in))
        h = h.at[0, dim].set(epsilon)
        
        X_plus = X + h
        X_minus = X - h
        
        f_center = model(X)
        f_plus = model(X_plus)
        f_minus = model(X_minus)
        
        second_deriv = (f_plus - 2 * f_center + f_minus) / (epsilon ** 2)
        curvatures = curvatures + jnp.sum(jnp.abs(second_deriv), axis=1)
    
    return curvatures


def run_training(
    func_name,
    architecture,
    method,
    seed,
    num_epochs=2000,
    learning_rate=0.001,
    grid_schedule=None,
    n_train=4000,
    verbose=False
):
    
    start_time = time.time()
    
    # Get function info
    function = func_dict[func_name]
    n_in = func_dims[func_name]
    
    # Generate training data (uniform random sampling)
    x, y = generate_feyn_data(function, n_in, 5000, seed)

    # Split data, only for uniformity with previous experiments, we don't use the test set anywhere
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=seed)
    
    # Create model
    req_params = {'k': 3, 'G': 3, 'init_scheme': {'type': 'glorot_fine'}}
    model = KAN(
        layer_dims=architecture,
        layer_type='spline',
        required_parameters=req_params,
        seed=seed
    )
    
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
    
    # Setup adaptation framework
    state = AdaptationState()
    trigger = ScheduledTrigger(epochs=list(grid_schedule.keys()))
    
    # Configure IDF and strategy based on method
    if method == 'input_adaptive':
        idf = UniformDensity()
        strategy = MixedAdaptation(grid_e=0.0)  # Quantile-based on input distribution
    elif method == 'curvature_adaptive':
        idf = CurvatureDensity()
        strategy = MixedAdaptation(grid_e=0.0)  # Quantile-based on curvature
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Training loop
    train_losses = []
    
    for epoch in range(num_epochs):
        # Check for grid adaptation
        if trigger(state):
            G_new = grid_schedule[epoch]
            
            if verbose:
                print(f"  Epoch {epoch}: Adapting grid to G={G_new}")
            
            # Compute curvatures if needed
            if method == 'curvature_adaptive':
                curvatures = compute_curvature(model, X_train)
                model.update_grids(
                    X_train,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new,
                    curvatures=curvatures
                )
            else:
                model.update_grids(
                    X_train,
                    idf=idf,
                    strategy=strategy,
                    grid_size_new=G_new
                )
            
            # Reset optimizer after grid change
            optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)
            state = reset_after_adaptation(state)
        
        # Training step
        loss = train_step(model, optimizer, X_train, y_train)
        train_losses.append(float(loss))
        
        state = update_state(state, loss=loss)
    
    # Final evaluation on fine grid
    rel_l2_error = feyn_fit_eval(model, function, n_in)
    
    wall_time = time.time() - start_time
    
    if verbose:
        print(f"  Relative L^2 error: {rel_l2_error:.6e} (time: {wall_time:.1f}s)")
    
    return {
        'rel_l2_error': rel_l2_error,
        'train_losses': np.array(train_losses),
        'wall_time': wall_time
    }

## Experiment Configuration

In [ ]:
func_dict = {"f1": f1, "f2": f2, "f3": f3, "f4": f4, "f5": f5, "f6": f6, "f7": f7, "f8": f8, "f9": f9, "f10": f10,
             "f11": f11, "f12": f12, "f13": f13, "f14": f14, "f15": f15}

func_dims = {"f1": 3, "f2": 2, "f3": 2, "f4": 2, "f5": 2, "f6": 2, "f7": 3, "f8": 2,
             "f9": 2, "f10": 2, "f11": 3, "f12": 3, "f13": 2, "f14": 3, "f15": 3}

# Architecture configuration
# Note: n_in will be set dynamically based on function
ARCHITECTURE = lambda n_in: [n_in, 10, 1]

# Methods to compare
METHODS = ['input_adaptive', 'curvature_adaptive']

# Number of random seeds
N_SEEDS = 3
SEEDS = list(range(44, 44 + N_SEEDS))

# Training configuration
TRAINING_CONFIG = {
    'num_epochs': 2000,
    'learning_rate': 1e-3,
    'grid_schedule': {0: 3, 500: 6, 1000: 9, 1500: 12},
    'n_train': 4000
}

# Print configuration summary
print("Benchmark Configuration:")
print(f"  Functions: {list(func_dict.keys())}")
print(f"  Methods: {METHODS}")
print(f"  Seeds: {N_SEEDS}")
print(f"\nTraining Config:")
print(f"  Epochs: {TRAINING_CONFIG['num_epochs']}")
print(f"  Grid schedule: {TRAINING_CONFIG['grid_schedule']}")
print(f"  Training samples: {TRAINING_CONFIG['n_train']}")
print(f"\nTotal experiments: {len(func_dict.keys())} × {len(METHODS)} × {N_SEEDS}")
print(f"  = {len(func_dict.keys()) * len(METHODS) * N_SEEDS} training runs")

## Run Benchmark


In [ ]:
# Storage for results
results = []

total_runs = len(func_dict.keys()) * len(METHODS) * N_SEEDS
current_run = 0

print(f"Starting benchmark: {total_runs} total runs")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

for func_name in func_dict.keys():
    print(f"Running Experiments for {func_name}.")
    
    architecture = ARCHITECTURE(func_dims[func_name])
    
    for method in METHODS:
        print(f"    Method: {method}", end=" ")
        
        method_results = []
        
        for seed in SEEDS:
            current_run += 1
            
            result = run_training(
                func_name=func_name,
                architecture=architecture,
                method=method,
                seed=seed,
                **TRAINING_CONFIG,
                verbose=False
            )
            
            # Store result
            results.append({
                'func_name': func_name,
                'n_in': func_dims[func_name],
                'arch_str': str(architecture),
                'method': method,
                'seed': seed,
                'rel_l2_error': result['rel_l2_error'],
                'wall_time': result['wall_time']
            })
            
            method_results.append(result['rel_l2_error'])
        
        # # Print summary for this method
        med_error = np.median(method_results)
        std_error = np.std(method_results)
        print(f"→ Med L^2: {med_error:.3e} ± {std_error:.3e} \t [{current_run}/{total_runs}]")

print("\n" + "=" * 80)
print(f"Benchmark complete at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total runs: {len(results)}")

## Save Results

In [ ]:
# Convert to DataFrame
df_results = pd.DataFrame(results)

# Save to CSV
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'results/feynman_results_{timestamp}.csv'
df_results.to_csv(filename, index=False)
print(f"Results saved to: {filename}")
